# Step 2: Test the Fine-tuned Model

Run this notebook AFTER step1_train.ipynb has completed.

What this notebook does:
1. Loads the saved LoRA adapter from Google Drive
2. Runs all 24 test questions through the model
3. Extracts the predicted answer letter (A/B/C/D)
4. Computes accuracy against ground truth
5. Prints a full results table

Before running: make sure Runtime > Change runtime type > GPU is set.

In [ ]:
# CELL 1: Install dependencies
!pip install unsloth --quiet
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps trl peft accelerate bitsandbytes --quiet
print("Installation complete.")

In [ ]:
# CELL 2: Mount Google Drive to load the saved adapter
from google.colab import drive
drive.mount('/content/drive')

import os
adapter_path = '/content/drive/MyDrive/qwen3_ioai_lora'
print("Files in adapter folder:", os.listdir(adapter_path))

In [ ]:
# CELL 3: Load the base model + LoRA adapter
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,   # Load from the saved adapter directory
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)  # Enable faster inference
print("Model loaded and ready for inference.")

In [ ]:
# CELL 4: Define the 24 test questions
# These are the held-out examples (indices 145-168 from the original dataset)
# They were NOT included in train.jsonl

TEST_QUESTIONS = [
    {
        "id": "ioai_rl_001",
        "question": "In Q-learning, the Q-value update rule is Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]. What does the term in square brackets represent?\nA) The learning rate\nB) The temporal difference (TD) error\nC) The discount factor\nD) The reward function",
        "answer": "B"
    },
    {
        "id": "ioai_rl_002",
        "question": "Which of the following best describes the exploration-exploitation tradeoff in reinforcement learning?\nA) Choosing between supervised and unsupervised learning\nB) Balancing trying new actions to discover rewards vs. using known good actions\nC) Deciding between deep and shallow neural networks\nD) Choosing between on-policy and off-policy methods",
        "answer": "B"
    },
    {
        "id": "ioai_rl_003",
        "question": "In the epsilon-greedy strategy with epsilon=0.1, what fraction of the time does the agent take a random action?\nA) 90%\nB) 1%\nC) 10%\nD) 50%",
        "answer": "C"
    },
    {
        "id": "ioai_rl_004",
        "question": "Policy gradient methods directly optimize the policy by:\nA) Building a complete model of the environment\nB) Computing gradients of expected cumulative reward with respect to policy parameters\nC) Using dynamic programming on the value function\nD) Clustering states by reward similarity",
        "answer": "B"
    },
    {
        "id": "ioai_dl_015",
        "question": "Layer Normalization differs from Batch Normalization in that it:\nA) Normalizes across the batch dimension\nB) Normalizes across the feature dimension for each sample independently\nC) Requires a large batch size to work well\nD) Can only be applied to convolutional layers",
        "answer": "B"
    },
    {
        "id": "ioai_dl_016",
        "question": "The vanishing gradient problem in deep networks is LEAST likely to occur when using which activation function?\nA) Sigmoid\nB) Tanh\nC) ReLU\nD) Softmax",
        "answer": "C"
    },
    {
        "id": "ioai_ml_020",
        "question": "You are building a spam classifier. False positives (legitimate emails marked as spam) are much more costly than false negatives. Which metric should you primarily optimize?\nA) Recall\nB) Precision\nC) F1-Score\nD) Accuracy",
        "answer": "B"
    },
    {
        "id": "ioai_ml_021",
        "question": "Gradient boosting builds an ensemble by:\nA) Training all trees simultaneously on bootstrap samples\nB) Training each new tree to predict the residual errors of the current ensemble\nC) Averaging predictions from randomly initialized trees\nD) Selecting the single best decision tree from many candidates",
        "answer": "B"
    },
    {
        "id": "ioai_ml_022",
        "question": "Which of the following is NOT an assumption of linear regression?\nA) Linearity between features and target\nB) Independence of residuals\nC) Homoscedasticity (constant variance of residuals)\nD) Features must follow a normal distribution",
        "answer": "D"
    },
    {
        "id": "ioai_ml_023",
        "question": "In hierarchical clustering, what does the dendrogram represent?\nA) The optimal number of clusters K\nB) A tree diagram showing the sequence of merges or splits of clusters\nC) The decision boundary of the clustering algorithm\nD) The distribution of feature values",
        "answer": "B"
    },
    {
        "id": "ioai_dl_017",
        "question": "What is the purpose of the forget gate in an LSTM?\nA) To add new information to the cell state\nB) To decide what information to remove from the cell state\nC) To control the output from the cell state\nD) To initialize the hidden state to zero",
        "answer": "B"
    },
    {
        "id": "ioai_dl_018",
        "question": "In a GAN (Generative Adversarial Network), the generator is trained to:\nA) Minimize the discriminator loss directly\nB) Produce outputs that the discriminator classifies as real\nC) Maximize the discriminator's accuracy\nD) Reconstruct the input from a compressed latent space",
        "answer": "B"
    },
    {
        "id": "ioai_nlp_001",
        "question": "Word2Vec's Skip-gram model is trained to:\nA) Predict the center word given surrounding context words\nB) Predict surrounding context words given the center word\nC) Classify the sentiment of a sentence\nD) Translate words between languages",
        "answer": "B"
    },
    {
        "id": "ioai_nlp_002",
        "question": "BERT is pre-trained using which two objectives?\nA) Next Sentence Prediction and Masked Language Modeling\nB) Causal Language Modeling and Sentence Classification\nC) Machine Translation and Question Answering\nD) Named Entity Recognition and Part-of-Speech Tagging",
        "answer": "A"
    },
    {
        "id": "ioai_nlp_003",
        "question": "The attention mechanism in transformers allows the model to:\nA) Process sequences in parallel rather than sequentially\nB) Compress the input into a fixed-size vector\nC) Apply the same filter to every position like a CNN\nD) Use recurrence to maintain a hidden state",
        "answer": "A"
    },
    {
        "id": "ioai_ml_024",
        "question": "What is the curse of dimensionality in machine learning?\nA) Neural networks becoming too deep to train\nB) As dimensions increase, data becomes increasingly sparse and distance metrics lose meaning\nC) Having too many training examples for memory\nD) The computational cost of matrix multiplication",
        "answer": "B"
    },
    {
        "id": "ioai_ml_025",
        "question": "DBSCAN clustering identifies clusters based on:\nA) Minimizing within-cluster variance\nB) Density of points, connecting core points within eps distance\nC) Hierarchical merging based on linkage criterion\nD) Maximizing the silhouette score directly",
        "answer": "B"
    },
    {
        "id": "ioai_dl_019",
        "question": "Dropout regularization during training randomly sets neuron outputs to zero. During inference:\nA) Dropout is applied at the same rate\nB) Dropout is disabled and neuron weights are scaled to account for it\nC) Dropout rate is halved\nD) Only the last layer uses dropout",
        "answer": "B"
    },
    {
        "id": "ioai_ml_026",
        "question": "Which of the following describes a Type I error in hypothesis testing?\nA) Failing to reject a false null hypothesis\nB) Rejecting a true null hypothesis\nC) A model with high variance\nD) A model with high bias",
        "answer": "B"
    },
    {
        "id": "ioai_dl_020",
        "question": "In the transformer positional encoding using sine and cosine functions, why are these specific functions used?\nA) They are computationally cheap\nB) They allow the model to generalize to sequence lengths longer than those seen in training\nC) They ensure all position encodings sum to one\nD) They guarantee orthogonality between all positions",
        "answer": "B"
    },
    {
        "id": "ioai_ml_027",
        "question": "Stochastic Gradient Descent (SGD) differs from full-batch gradient descent in that SGD:\nA) Uses the entire dataset to compute each gradient update\nB) Uses one or a small batch of samples per update, introducing noise that can help escape local minima\nC) Always converges to the global minimum\nD) Does not use a learning rate",
        "answer": "B"
    },
    {
        "id": "ioai_ml_028",
        "question": "What does the ROC curve plot?\nA) Precision vs. Recall at various thresholds\nB) True Positive Rate vs. False Positive Rate at various classification thresholds\nC) Training loss vs. validation loss over epochs\nD) Feature importance vs. model complexity",
        "answer": "B"
    },
    {
        "id": "ioai_ml_029",
        "question": "Which normalization technique is most appropriate when you know the data has outliers and you want to bound it to a fixed range like [0, 1]?\nA) Z-score standardization\nB) Min-Max scaling\nC) Log transformation\nD) Robust scaling using median and IQR",
        "answer": "D"
    },
    {
        "id": "ioai_ml_030",
        "question": "In multi-task learning, a model is trained on multiple tasks simultaneously. The primary benefit is:\nA) Reduced inference time compared to single-task models\nB) Shared representations that allow tasks to regularize each other and improve generalization\nC) Elimination of the need for labeled data\nD) Guaranteed improvement on all tasks simultaneously",
        "answer": "B"
    }
]

print(f"Total test questions: {len(TEST_QUESTIONS)}")

In [ ]:
# CELL 5: Inference function
import re
import torch

def get_model_answer(question_text, max_new_tokens=300):
    """
    Run a question through the fine-tuned model.
    Returns the full response string.
    """
    messages = [{"role": "user", "content": question_text}]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens = max_new_tokens,
            temperature = 0.1,
            do_sample = True,
            pad_token_id = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens = True
    )
    return response.strip()


def extract_letter(response):
    """
    Extract the answer letter (A, B, C, or D) from the model's response.
    Tries several patterns in order of confidence.
    """
    # Pattern 1: "The correct answer is X"
    m = re.search(r'correct answer is\s*([A-D])', response, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Pattern 2: "answer is X" or "answer: X"
    m = re.search(r'answer[\s:is]*([A-D])\)', response, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Pattern 3: Response starts with a letter like "B)" or "B."
    m = re.match(r'^\s*([A-D])[\)\.\s]', response)
    if m:
        return m.group(1).upper()

    # Pattern 4: Any standalone capital A/B/C/D in the first 50 chars
    m = re.search(r'\b([A-D])\b', response[:50])
    if m:
        return m.group(1).upper()

    return "UNKNOWN"


print("Inference functions defined.")

In [ ]:
# CELL 6: Run all 24 test questions and collect results
results = []

for i, item in enumerate(TEST_QUESTIONS):
    print(f"[{i+1}/{len(TEST_QUESTIONS)}] Testing {item['id']}...", end=" ")

    response = get_model_answer(item['question'])
    predicted = extract_letter(response)
    correct = (predicted == item['answer'])

    results.append({
        'id': item['id'],
        'question_preview': item['question'][:80] + '...',
        'ground_truth': item['answer'],
        'predicted': predicted,
        'correct': correct,
        'full_response': response
    })

    status = "CORRECT" if correct else f"WRONG (predicted {predicted}, expected {item['answer']})"
    print(status)

print("\nAll questions tested.")

In [ ]:
# CELL 7: Compute and display accuracy
total = len(results)
correct_count = sum(1 for r in results if r['correct'])
accuracy = correct_count / total * 100

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Total questions : {total}")
print(f"Correct         : {correct_count}")
print(f"Wrong           : {total - correct_count}")
print(f"Accuracy        : {accuracy:.1f}%")
print("=" * 60)

In [ ]:
# CELL 8: Detailed results table
print(f"{'No.':<4} {'ID':<20} {'Truth':<7} {'Pred':<7} {'Result':<10}")
print("-" * 55)

for i, r in enumerate(results):
    status = "CORRECT" if r['correct'] else "WRONG"
    print(f"{i+1:<4} {r['id']:<20} {r['ground_truth']:<7} {r['predicted']:<7} {status:<10}")

print("-" * 55)
print(f"Accuracy: {accuracy:.1f}%")

In [ ]:
# CELL 9: Show wrong answers with model reasoning for analysis
wrong = [r for r in results if not r['correct']]

if not wrong:
    print("Perfect score. No wrong answers.")
else:
    print(f"Wrong answers ({len(wrong)} total):")
    print("=" * 60)
    for r in wrong:
        print(f"\nID: {r['id']}")
        print(f"Question: {r['question_preview']}")
        print(f"Expected: {r['ground_truth']}")
        print(f"Predicted: {r['predicted']}")
        print(f"Model response: {r['full_response'][:400]}")
        print("-" * 40)

In [ ]:
# CELL 10: Save results to CSV in Google Drive
import csv

csv_path = '/content/drive/MyDrive/qwen3_ioai_test_results.csv'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'ground_truth', 'predicted', 'correct', 'full_response'])
    writer.writeheader()
    for r in results:
        writer.writerow({
            'id': r['id'],
            'ground_truth': r['ground_truth'],
            'predicted': r['predicted'],
            'correct': r['correct'],
            'full_response': r['full_response']
        })

print(f"Results saved to: {csv_path}")
print(f"Final accuracy: {accuracy:.1f}%")